# Multi-Class Warehouse Object Detection: YOLO11s vs. YOLO11s-SS2D
## 3-Arm Comparative Convergence Protocol (Google Colab / Kaggle GPU)

This notebook runs the 3 experimental arms under strictly matched hyperparameters:
- **Arm 1: Control Baseline** — YOLO11s (COCO Pretrained `yolo11s.pt`)
- **Arm 2: Case B Single-Stage** — YOLO11s-SS2D (`yolo11s_ss2d_full_warmstart.pt`), unfrozen end-to-end
- **Arm 3: Case B Two-Stage** — YOLO11s-SS2D (`yolo11s_ss2d_full_warmstart.pt`), Stage 1 Warmup (`freeze=10`) + Stage 2 Full Fine-Tuning

**Hardware:** NVIDIA T4 / V100 / A100 GPU (CUDA accelerated)
**Dataset:** Zero-leakage multi-camera warehouse benchmark (`smartspaces_micro`)

In [ ]:
# 1. Environment Setup & GPU Check
!nvidia-smi
!pip install -q ultralytics torchvision opencv-python

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

In [ ]:
# 2. Register Custom SS2D Layer with Ultralytics
import os
import sys

# Ensure project root is in path
sys.path.insert(0, os.path.abspath("."))

import ultralytics.nn.tasks as tasks
import ultralytics.nn.modules as modules
from models.warehouse_ss2d import WarehouseSS2DBlock

tasks.WarehouseSS2DBlock = WarehouseSS2DBlock
modules.WarehouseSS2DBlock = WarehouseSS2DBlock
print("✅ WarehouseSS2DBlock registered successfully with Ultralytics.")

In [ ]:
# 3. Run Arm 1: Control Baseline YOLO11s (20 Epochs)
!python scripts/run_two_stage_experiment.py --arm 1 --epochs 20 --device 0

In [ ]:
# 4. Run Arm 2: Case B Single-Stage Direct Fine-Tuning (20 Epochs)
!python scripts/run_two_stage_experiment.py --arm 2 --epochs 20 --device 0

In [ ]:
# 5. Run Arm 3: Case B Two-Stage Protocol (5 Epoch Warmup + 15 Epoch Fine-Tune)
!python scripts/run_two_stage_experiment.py --arm 3 --epochs 20 --device 0

In [ ]:
# 6. Results Summary and Validation Metrics Comparison
import pandas as pd
import glob

results = {}
for path in glob.glob("runs/detect/runs/two_stage_protocol_study/*/results.csv"):
    arm_name = path.split("/")[-2]
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    last_row = df.iloc[-1]
    results[arm_name] = {
        "mAP50": last_row.get("metrics/mAP50(B)", 0.0),
        "mAP50-95": last_row.get("metrics/mAP50-95(B)", 0.0),
        "box_loss": last_row.get("val/box_loss", 0.0),
        "cls_loss": last_row.get("val/cls_loss", 0.0)
    }

summary_df = pd.DataFrame(results).T
print("\n=================== EMPIRICAL CONVERGENCE SUMMARY ===================")
print(summary_df)
summary_df.to_csv("gpu_convergence_summary.csv")